# MICrONS PCA — Option 2: Trial-averaged PCA across cortical areas

Applying PCA per cortical area to trial-averaged responses, asking whether the three stimulus classes (Clip, Monet2, Trippy) occupy distinct regions of population state space and whether the strength of that separation differs across areas (V1, AL, LM, RL).

See `docs/specs/2026-05-02-pca-design.md` for the design and `docs/plans/2026-05-02-pca-implementation.md` for the implementation plan.

## How to use this notebook

The pipeline is **session-swappable**: every session-specific value is read from data, and outputs are partitioned by session subdirectory. To run on a different session, change `SESSION` in the configuration block below and restart the kernel.

## Part 0 — Setup

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.io as pio

import microns_eda
import option2_pca_utils as pca_utils

In [ ]:
# === Configuration block — every tunable lives here. ===
SESSION = "7_5"

# Preprocessing
N_FRAMES_TRUNCATE = 75              # Clip trial length; truncates Monet2/Trippy from 113.
TREADMILL_OUTLIER_THRESHOLD = 1.0   # Same threshold as EDA's running-outlier filter.

# PCA / metrics
N_COMPONENTS = 10
N_BALANCE_REPLICATES = 20           # subsamples per balanced silhouette.
N_SHUFFLES = 100                    # for both silhouette and classifier nulls.
N_POPULATION_SUBSAMPLES = 20        # for the equal-population control.
N_FOLDS_CV = 5

# Reproducibility
RANDOM_SEED = 42

# Paths
DATADIR = Path(os.environ.get("MICRONS_DATADIR", "../neuroscience"))
FIGURES_DIR = Path(f"figures/option2/{SESSION}")
RESULTS_DIR = Path(f"results/option2/{SESSION}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)

print(f"SESSION       = {SESSION}")
print(f"DATADIR       = {DATADIR}")
print(f"FIGURES_DIR   = {FIGURES_DIR}")
print(f"RESULTS_DIR   = {RESULTS_DIR}")

### Sanity check

We open the dataset, verify session `{SESSION}` is loadable, recompute `clean_trial_indices` from the EDA's running-outlier rule, and print the per-class composition. **All session-specific values are read from data** — switching `SESSION` changes the printed numbers but no code.

This cell fails loudly if the environment is wrong (path, library versions, missing session). Pass = safe to proceed.

In [ ]:
# === Sanity check ===
reader = microns_eda.open_dataset(DATADIR)
sessions = microns_eda.list_sessions(DATADIR)
assert SESSION in sessions, f"session {SESSION!r} not in {sessions}"
print(f"Found {len(sessions)} sessions; using {SESSION!r}.")

meta = microns_eda.get_session_meta(DATADIR, SESSION)
print(f"  n_neurons (read from data) = {meta['n_neurons']}")

# Build per-trial treadmill means via the same path as EDA.
n_trials = meta["n_trials"]
per_trial_tread_means = np.empty(n_trials, dtype=np.float64)
for i in range(n_trials):
    trial = microns_eda.load_trial(reader, DATADIR, SESSION, i)
    per_trial_tread_means[i] = np.nanmean(trial["treadmill"])

clean_trial_indices, running_mask = microns_eda.compute_clean_trial_indices(
    per_trial_tread_means, threshold=TREADMILL_OUTLIER_THRESHOLD,
)

# Stim labels for the clean subset.
stim_map = microns_eda.build_stim_type_map(reader, SESSION)
stim_types_per_trial = np.array([
    stim_map[h.decode() if isinstance(h, bytes) else h]
    for h in meta["condition_hashes"]
])
labels = stim_types_per_trial[clean_trial_indices]

# Read class counts from data — never hard-code.
class_counts = pd.Series(labels).value_counts().to_dict()
print(f"  n_trials (total)            = {n_trials}")
print(f"  n_trials (after running QC) = {len(clean_trial_indices)}")
print(f"  per-class breakdown (clean):")
for k in sorted(class_counts):
    print(f"    {k:8s} {class_counts[k]}")

assert len(clean_trial_indices) > 0, "No clean trials remain — threshold too low?"
assert len(class_counts) >= 2, "Need at least 2 stim classes for separability."
print("Sanity check passed.")

## Part 1 — Preprocess and build per-area matrices

We load the full-session response matrix, apply the locked-in preprocessing recipe, then collapse each clean trial to one vector per cortical area.

**What's locked in (see spec):**

- **Detrend.** Photobleaching makes the calcium indicator dimmer over the course of the recording (the EDA measured a ~45% decrease across `7_5`). Without correction, the slow fade dominates the first principal component and drowns out anything stimulus-related. We subtract a per-neuron linear fit to remove the fade while leaving moment-to-moment activity untouched.
- **Z-score per neuron.** Different neurons fluoresce at different baseline brightness. After z-scoring (subtract mean, divide by std), every neuron contributes on the same scale, so PCA isn't dominated by the brightest few neurons.
- **Truncate to 75 frames.** Clip trials are 75 frames; Monet2 and Trippy are 113. Truncating Monet2/Trippy makes the trial-averaged vectors directly comparable across stimulus classes.
- **Trial-average across time.** Each trial collapses to a single vector summarising its average activity per neuron.
- **One matrix per cortical area.** V1, AL, LM, RL are analysed independently — each area's PCA is fit on its own column subset.

We expect: four matrices with `len(clean_trial_indices) = 453` rows for `7_5` and column counts matching the EDA cohort table (V1 ≈ 5,485, LM ≈ 1,262, RL ≈ 1,033, AL ≈ 414).

In [ ]:
# Load the full neuron × time matrix and trial boundaries.
print("loading full session responses...")
responses, trial_boundaries, _ = microns_eda.load_session_responses(
    reader, DATADIR, SESSION
)
print(f"  responses shape       = {responses.shape}")
print(f"  trial_boundaries len  = {len(trial_boundaries)}")
print(f"  total timesteps       = {responses.shape[1]}")

In [ ]:
# Stage A — full-timeseries preprocessing (log? -> detrend -> z-score).
responses_pp = pca_utils.preprocess_responses(responses, apply_log=False)
assert responses_pp.shape == responses.shape
print(f"preprocess_responses applied; shape unchanged at {responses_pp.shape}.")

# Stage B — trial-level: truncate, mask, per-area split + average.
per_area = pca_utils.build_per_area_matrices(
    responses_pp,
    trial_boundaries=trial_boundaries,
    clean_trial_indices=clean_trial_indices,
    brain_areas=meta["brain_areas"],
    n_frames=N_FRAMES_TRUNCATE,
)

print(f"\nper-area matrices ({len(clean_trial_indices)} clean trials):")
total_neurons = 0
for area in sorted(per_area.keys()):
    X = per_area[area]
    total_neurons += X.shape[1]
    print(f"  {area:5s}: shape {X.shape!s:18s} "
          f"(NaN: {np.isnan(X).sum()}; zero-variance cols: {(X.std(axis=0) == 0).sum()})")
print(f"  total neurons = {total_neurons} (matches meta: {total_neurons == meta['n_neurons']})")

# Sanity asserts.
for area, X in per_area.items():
    assert X.shape[0] == len(clean_trial_indices), f"{area} row count wrong"
    assert not np.isnan(X).any(), f"{area} has NaNs"
    # Constant columns would break PCA / silhouette; warn rather than assert.
    if (X.std(axis=0) == 0).any():
        print(f"  WARNING: {area} has constant column(s) — PCA will treat as zero-variance.")

## Part 2 — PCA on V1

V1 is our reference area: it dominates the population (5,485 / 8,194 neurons) and its low-level feature tuning means we expect strong stimulus-class separation. We fit PCA on the trial-averaged matrix `X_V1`, visualise the top-3 PCs as a 2-D scatter (for the report) and a 3-D Plotly scatter (for the oral presentation), and inspect the scree plot.

**What we're looking for:**

- 2-D scatter: three colour clouds with different centroids; some overlap is expected.
- 3-D scatter: separation that wasn't visible in 2-D may resolve along PC3.
- Scree plot: how much of the total variance lives in the top 3 PCs we're visualising. A high cumulative percentage means we're looking at the dominant signal; a low percentage means we're looking at a peripheral one.

In [ ]:
pca_V1, X_V1_pcs = pca_utils.fit_pca(
    per_area["V1"], n_components=N_COMPONENTS, random_state=RANDOM_SEED,
)
print(f"V1 PCA fitted; X_pcs shape = {X_V1_pcs.shape}")
print(f"  top-3 cumulative variance = {100 * pca_V1.explained_variance_ratio_[:3].sum():.1f}%")
print(f"  top-10 cumulative         = {100 * pca_V1.explained_variance_ratio_[:10].sum():.1f}%")

In [ ]:
# 2-D scatter (matplotlib, for the report).
fig, ax = plt.subplots(figsize=(7, 5.5))
pca_utils.plot_pca_2d(X_V1_pcs, labels, area_name="V1", pca=pca_V1, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "2_pca_v1_2d.png", dpi=150)
plt.show()

In [ ]:
# Scree plot.
fig, ax = plt.subplots(figsize=(8, 4.5))
pca_utils.plot_scree(pca_V1, area_name="V1", n_show=20, ax=ax)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "2_pca_v1_scree.png", dpi=150)
plt.show()

In [ ]:
# 3-D Plotly scatter (interactive; saved to HTML for the oral presentation).
fig3d_v1 = pca_utils.plot_pca_3d_plotly(
    X_V1_pcs, labels, area_name="V1", pca=pca_V1
)
out_html = FIGURES_DIR / "2_pca_v1_3d.html"
fig3d_v1.write_html(str(out_html))
print(f"saved {out_html}")
fig3d_v1.show()

### Interpretation — V1 (PC1–3)

*[Add a short paragraph after running: do the three classes cluster visibly? Is the separation concentrated in PC1–2 or does PC3 add information? What fraction of variance is in the top 3 PCs?]*

## Part 3 — Diagnostics on V1

### What is a silhouette score, and why do we balance it?

The **silhouette score** measures, for each trial, how tightly it sits with neighbours of the same stimulus class versus how far it sits from neighbours of other classes. The per-trial score ranges from −1 (worse than random; the trial sits closer to other classes) to +1 (perfectly clustered with same-class neighbours). The dataset-level silhouette is the average over trials. Higher means cleaner separation.

**Why balance.** In `7_5` Clip has 377 trials but Monet2 and Trippy have only 38 each. Raw silhouette gives Clip's compactness ten times more weight than the minority classes', which biases the score and makes it hard to interpret across areas. We instead compute the silhouette on a **balanced 38 / 38 / 38 subsample**, average over 20 such subsamples, and use that average as the observed value. The minority count is read from the data, so this works automatically on other sessions.

We compare the observed silhouette to a **null distribution** built by shuffling stimulus labels 100 times. If the observed silhouette exceeds the 95th percentile of the null (empirical p < 0.05), the cluster structure we see is unlikely to be a coincidence.

In [ ]:
print("computing balanced silhouette + null on V1 (this takes ~30-60 s)...")
sil_V1 = pca_utils.silhouette_balanced(
    X_V1_pcs[:, :3], labels,
    n_replicates=N_BALANCE_REPLICATES, seed=RANDOM_SEED,
)
sil_null_V1 = pca_utils.silhouette_balanced_null(
    X_V1_pcs[:, :3], labels,
    n_replicates=N_BALANCE_REPLICATES,
    n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 1,
)
sil_V1_p = float((1 + (sil_null_V1 >= sil_V1["observed"]).sum()) / (1 + N_SHUFFLES))
print(f"  observed silhouette  = {sil_V1['observed']:+.3f}  (n_per_class={sil_V1['n_per_class']})")
print(f"  null median (shuffle)= {np.median(sil_null_V1):+.3f}")
print(f"  empirical p          = {sil_V1_p:.3f}")

### Variance ≠ separability — why we add a classifier metric

PCA finds the directions of **highest variance** in the data. That isn't the same as the directions that **separate stimulus classes**. If the class signal lives in PC4 or PC5 instead of PC1–3, a silhouette in top-3 PC space will miss it.

We therefore add a complementary metric: **5-fold cross-validated multinomial logistic regression accuracy on the full trial-averaged matrix** (no PCA reduction). The classifier sees every neuron and finds the best linear combinations for separating Clip / Monet2 / Trippy, regardless of which PCs they happen to align with. The reference is the **majority-class baseline**: a classifier that always predicts Clip would get 377/453 ≈ 83.2% on `7_5`. Anything meaningfully above this is real signal.

When the two metrics agree (both significant or both not), the conclusion is robust. When they disagree (e.g., classifier > chance but silhouette ≈ null), it tells us *where* the signal lives — typically not in the top-3 variance directions.

In [ ]:
print("computing classifier + null on V1 (this takes ~60-120 s)...")
clf_V1 = pca_utils.classify_cv(
    per_area["V1"], labels, n_folds=N_FOLDS_CV, seed=RANDOM_SEED,
)
clf_null_V1 = pca_utils.classify_cv_null(
    per_area["V1"], labels,
    n_folds=N_FOLDS_CV, n_shuffles=N_SHUFFLES, seed=RANDOM_SEED + 2,
)
clf_V1_p = float((1 + (clf_null_V1 >= clf_V1["observed"]).sum()) / (1 + N_SHUFFLES))
print(f"  observed accuracy   = {clf_V1['observed']:.3f}")
print(f"  chance (majority)   = {clf_V1['chance']:.3f}")
print(f"  null median         = {np.median(clf_null_V1):.3f}")
print(f"  empirical p         = {clf_V1_p:.3f}")

In [ ]:
# Two histograms side-by-side.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pca_utils.plot_silhouette_with_null(
    sil_V1["observed"], sil_null_V1, area_name="V1", ax=axes[0],
)
pca_utils.plot_classifier_with_null(
    clf_V1["observed"], clf_null_V1, area_name="V1",
    chance=clf_V1["chance"], ax=axes[1],
)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "3_v1_diagnostics.png", dpi=150)
plt.show()

### Interpretation — V1 diagnostics

*[Add after running: do silhouette and classifier agree? If both p < 0.05, V1 separates the three stimuli reliably. If silhouette is null but classifier beats chance, the separation exists but isn't in the top-3 variance directions — informative for the time-resolved follow-up.]*